# 1. Prepare scaled VLA reference maps

Match the masked VLA cube to the FAST beam and grid to check source scale factors. Then construct the **native VLA resolution** surface brightness maps used by notebook 02. Run all cells in order from the repository root.

In [ ]:
import numpy as np
import astropy.units as u
from astropy.io import fits
from spectral_cube import SpectralCube
from radio_beam import Beam
from astropy.wcs import WCS
from reproject import reproject_interp
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.coordinates import SkyCoord

u.add_enabled_units(u.def_unit(['mJy/Beam'], represents=u.Jy/u.beam/1000))
# -------------------------
# Input files
# -------------------------
vla_cube_file = './VLA/HCG100_HI.fits.gz'
vla_mask_file = './VLA/HCG100_mask.fits.gz'
fast_cube_file = './rawcube/HCG100-2fov-HIcube-pix1.4arcmin-20kms-1-mJyBeam-1.fits'
fast_header = fits.getheader(fast_cube_file)  

# -------------------------
# Read VLA cube and mask
# -------------------------
vla_cube = SpectralCube.read(vla_cube_file).with_spectral_unit(
    u.km/u.s,
    velocity_convention='optical'
)

vla_mask_cube = SpectralCube.read(vla_mask_file).with_spectral_unit(
    u.km/u.s,
    velocity_convention='optical'
)

vla_masked = vla_cube.with_mask(vla_mask_cube > 0)

# -------------------------
# Read FAST cube as reference grid
# -------------------------
fast_cube = SpectralCube.read(fast_cube_file)

# -------------------------
# Restrict to relevant velocity range
# -------------------------
vmin = 5000 * u.km/u.s
vmax = 5800 * u.km/u.s

vla_sub = vla_masked.spectral_slab(vmin, vmax)
fast_sub = fast_cube.spectral_slab(vmin, vmax)

# -------------------------
# Convolve VLA cube to FAST final beam
# -------------------------
target_beam = Beam.from_fits_header(fast_header)#Beam(major=4*u.arcmin, minor=4*u.arcmin, pa=0*u.deg)

vla_smooth = vla_sub.convolve_to(target_beam)

# Convert Jy/beam to mJy/beam
vla_smooth_mjy = vla_smooth * 1000.0

print(vla_smooth_mjy)

In [ ]:
fast_sub

In [ ]:
vla_smooth_mjy.shape

In [ ]:


# FAST原始cube
fast_cube_file = './rawcube/HCG100-2fov-HIcube-pix1.4arcmin-20kms-1-mJyBeam-1.fits'

fast_cube = SpectralCube.read(fast_cube_file)

# 只取相关速度范围
vmin = 5000 * u.km/u.s
vmax = 5800 * u.km/u.s

fast_sub = fast_cube.spectral_slab(vmin, vmax)

fast_vel = fast_sub.spectral_axis.to(u.km/u.s).value
vla_vel = vla_smooth_mjy.spectral_axis.to(u.km/u.s).value

print('FAST velocity:', fast_vel[:5], fast_vel[-5:])
print('VLA velocity:', vla_vel[:5], vla_vel[-5:])
print('FAST shape:', fast_sub.shape)
print('VLA smooth shape:', vla_smooth_mjy.shape)

In [ ]:
from scipy.interpolate import interp1d

vla_data = vla_smooth_mjy.filled_data[:].value  # shape: nchan_vla, ny_vla, nx_vla

# 处理速度轴方向
sort_idx = np.argsort(vla_vel)
vla_vel_sorted = vla_vel[sort_idx]
vla_data_sorted = vla_data[sort_idx, :, :]

# 构建插值函数
interp_func = interp1d(
    vla_vel_sorted,
    vla_data_sorted,
    axis=0,
    bounds_error=False,
    fill_value=np.nan
)

# 插值到FAST速度轴
vla_spec_on_fastvel = interp_func(fast_vel)

print(vla_spec_on_fastvel.shape)

In [ ]:
# VLA 2D celestial WCS
vla_wcs_2d = vla_smooth_mjy.wcs.celestial

# FAST 2D celestial WCS
fast_wcs_2d = fast_sub.wcs.celestial

ny_fast, nx_fast = fast_sub.shape[1], fast_sub.shape[2]

print(ny_fast, nx_fast)

In [ ]:
vla_on_fast_data = np.full(
    (len(fast_vel), ny_fast, nx_fast),
    np.nan,
    dtype=float
)

for i in range(len(fast_vel)):
    image_vla = vla_spec_on_fastvel[i, :, :]

    reproj_img, footprint = reproject_interp(
        (image_vla, vla_wcs_2d),
        fast_wcs_2d,
        shape_out=(ny_fast, nx_fast)
    )

    # footprint==0的地方设为nan
    reproj_img[footprint == 0] = np.nan
    vla_on_fast_data[i, :, :] = reproj_img

print(vla_on_fast_data.shape)

In [ ]:
fast_header = fast_sub.header.copy()

# 数据单位
fast_header['BUNIT'] = 'mJy/beam'
fast_header['COMMENT'] = 'VLA cube convolved to the FAST header beam, interpolated in velocity and reprojected to the FAST grid.'

outname = './VLA/vla_smooth_3arcmin_on_fastgrid_5000_5800_mJyBeam.fits'

fits.writeto(
    outname,
    vla_on_fast_data.astype('float32'),
    fast_header,
    overwrite=True
)

print('Saved:', outname)

In [ ]:
test_cube = SpectralCube.read(outname).with_spectral_unit(
    u.km/u.s,
    velocity_convention='optical'
)

print(test_cube.shape)
print(test_cube.unit)
print(test_cube.spectral_axis[:5])

import matplotlib.pyplot as plt

v1 = 5200 * u.km/u.s
v2 = 5600 * u.km/u.s

fast_m0 = fast_sub.spectral_slab(v1, v2).moment0()
vla_m0_on_fast = test_cube.spectral_slab(v1, v2).moment0()

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(fast_m0.value, origin='lower')
plt.title('FAST raw moment0')
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(vla_m0_on_fast.value, origin='lower')
plt.title('VLA smooth on FAST grid')
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
vla_on_fast_cube = SpectralCube.read(
    './VLA/vla_smooth_3arcmin_on_fastgrid_5000_5800_mJyBeam.fits'
).with_spectral_unit(
    u.km/u.s,
    velocity_convention='optical'
)

# vla_on_fast_cube: VLA convolved + reprojected to FAST grid
# fast_sub: raw FAST cube on same grid

mom_ranges = {
    '5200_5400': (5200, 5400),
    '5400_5600': (5400, 5600),
    '5200_5600': (5200, 5600),
    '5000_5600': (5000, 5600),
}

for tag, (v1, v2) in mom_ranges.items():
    vmin = v1 * u.km/u.s
    vmax = v2 * u.km/u.s

    vla_m0 = vla_on_fast_cube.spectral_slab(vmin, vmax).moment0()
    fast_m0 = fast_sub.spectral_slab(vmin, vmax).moment0()

    h_vla = vla_m0.header.copy()
    h_vla['BUNIT'] = 'mJy/beam km/s'
    fits.writeto(
        f'./VLA/vla_smooth_on_fastgrid_M0_{tag}.fits',
        vla_m0.value.astype('float32'),
        h_vla,
        overwrite=True
    )

    h_fast = fast_m0.header.copy()
    h_fast['BUNIT'] = 'mJy/beam km/s'
    fits.writeto(
        f'./rawcube/fast_raw_M0_{tag}.fits',
        fast_m0.value.astype('float32'),
        h_fast,
        overwrite=True
    )

In [ ]:
sources = {
    'HCG100': {
        'ra': 0.3022699,
        'dec': 13.0891334,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 4.0,
    },
    'NGC7810': {
        'ra': 0.5763757,
        'dec': 12.9642823,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 2.2,
    },
    'MRK935': {
        'ra': 0.4504456,
        'dec': 13.1028654,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 1.75,
    },
    'AGC105092': {
        'ra': 0.4095431,
        'dec': 12.9043733,
        'vmin': 5200,
        'vmax': 5600,
        'radius_arcmin': 2.2,
    },
}

In [ ]:
# sources = {
#     'HCG100': {
#         'ra': 0.3022699,
#         'dec': 13.0891334,
#         'vmin': 5200,
#         'vmax': 5600,
#         'radius_arcmin': 3.5,
#     },
#     'NGC7810': {
#         'ra': 0.5763757,
#         'dec': 12.9642823,
#         'vmin': 5200,
#         'vmax': 5600,
#         'radius_arcmin': 2.0,
#     },
#     'MRK935': {
#         'ra': 0.4504456,
#         'dec': 13.1028654,
#         'vmin': 5200,
#         'vmax': 5600,
#         'radius_arcmin': 1.7,
#     },
#     'AGC105092': {
#         'ra': 0.4095431,
#         'dec': 12.9043733,
#         'vmin': 5200,
#         'vmax': 5600,
#         'radius_arcmin': 2.2,
#     },
# }

In [ ]:
from astropy.coordinates import SkyCoord

def make_circular_mask_from_wcs(wcs2d, shape, ra, dec, radius_arcmin):
    ny, nx = shape
    yy, xx = np.mgrid[:ny, :nx]

    center = SkyCoord(ra*u.deg, dec*u.deg)
    x0, y0 = wcs2d.world_to_pixel(center)

    # 估算pixel scale，适用于小视场
    pixscale_y = np.abs(wcs2d.proj_plane_pixel_scales()[1].to(u.arcmin).value)
    pixscale_x = np.abs(wcs2d.proj_plane_pixel_scales()[0].to(u.arcmin).value)
    r_pix = radius_arcmin / np.sqrt(pixscale_x * pixscale_y)

    rr = np.sqrt((xx - x0)**2 + (yy - y0)**2)
    return rr <= r_pix

In [ ]:
def source_peak_ratio(fast_cube, vla_cube, source, wcs2d):
    vmin = source['vmin'] * u.km/u.s
    vmax = source['vmax'] * u.km/u.s

    fast_m0 = fast_cube.spectral_slab(vmin, vmax).moment0().value
    vla_m0 = vla_cube.spectral_slab(vmin, vmax).moment0().value

    mask = make_circular_mask_from_wcs(
        wcs2d,
        fast_m0.shape,
        source['ra'],
        source['dec'],
        source['radius_arcmin']
    )

    fast_peak = np.nanmax(fast_m0[mask])
    vla_peak = np.nanmax(vla_m0[mask])
    ratio = fast_peak / vla_peak

    fast_sum = np.nansum(fast_m0[mask])
    vla_sum = np.nansum(vla_m0[mask])
    aperture_ratio = fast_sum / vla_sum

    return fast_peak, vla_peak, ratio, aperture_ratio

In [ ]:
for name, source in sources.items():
    fast_peak, vla_peak, peak_ratio, aperture_ratio = source_peak_ratio(
        fast_sub,
        vla_on_fast_cube,
        source,
        fast_wcs_2d
    )
    print(
        name,
        'FAST peak =', fast_peak,
        'VLA peak =', vla_peak,
        'peak ratio =', peak_ratio,
        'aperture ratio =', aperture_ratio
    )

In [ ]:
scale_factors = {
    'HCG100': 1.2846954129518,
    'NGC7810': 1.1769393092739915,
    'MRK935': 1.0799621913039583,
    'AGC105092': 1.4756065799816787,
}

In [ ]:
vla_data = vla_on_fast_cube.filled_data[:].value
vel = vla_on_fast_cube.spectral_axis.to(u.km/u.s).value

nchan, ny, nx = vla_data.shape
truth_total = np.zeros_like(vla_data, dtype=float)

truth_components = {}

for name, src in sources.items():
    spatial_mask = make_circular_mask_from_wcs(
        fast_wcs_2d,
        (ny, nx),
        src['ra'],
        src['dec'],
        src['radius_arcmin']
    )

    vel_mask = (vel >= src['vmin']) & (vel <= src['vmax'])

    comp = np.zeros_like(vla_data, dtype=float)

    comp[vel_mask, :, :] = (
        vla_data[vel_mask, :, :]
        * spatial_mask[None, :, :]
        * scale_factors[name]
    )

    truth_components[name] = comp
    truth_total += comp

    out_header = vla_on_fast_cube.header.copy()
    out_header['BUNIT'] = 'mJy/beam'
    fits.writeto(
        f'./VLA/truth_{name}_scaled_VLA_on_FASTgrid.fits',
        comp.astype('float32'),
        out_header,
        overwrite=True
    )

fits.writeto(
    './VLA/truth_total_scaled_VLA_on_FASTgrid.fits',
    truth_total.astype('float32'),
    vla_on_fast_cube.header,
    overwrite=True
)

In [ ]:
w = WCS(fast_header).celestial

def skycoord_to_pixel(wcs2d, ra_deg, dec_deg):
    coord = SkyCoord(ra_deg*u.deg, dec_deg*u.deg)
    x, y = wcs2d.world_to_pixel(coord)
    return x, y

def arcmin_to_pixel_radius(wcs2d, radius_arcmin):
    pixscale = u.Quantity(wcs2d.proj_plane_pixel_scales(), u.deg)
    pixscale_x = np.abs(pixscale[0].to(u.arcmin).value)
    pixscale_y = np.abs(pixscale[1].to(u.arcmin).value)
    return radius_arcmin / np.sqrt(pixscale_x * pixscale_y)

for name, src in sources.items():
    x, y = skycoord_to_pixel(w, src['ra'], src['dec'])
    r_pix = arcmin_to_pixel_radius(w, src['radius_arcmin'])

    src['xpix'] = x
    src['ypix'] = y
    src['rpix'] = r_pix
    src['aperture'] = CircularAperture((x, y), r=r_pix)

    print(name, 'x=', x, 'y=', y, 'r_pix=', r_pix)

In [ ]:
truth_cube = SpectralCube.read(
    './VLA/truth_total_scaled_VLA_on_FASTgrid.fits'
).with_spectral_unit(
    u.km/u.s,
    velocity_convention='optical'
)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(fast_m0.value, origin='lower',vmin=0)
plt.title('FAST raw moment0')
plt.colorbar()

#plt.subplot(1,2,2)

for tag, v1, v2 in [
    ('5200_5600', 5200, 5600),
]:
    m0 = truth_cube.spectral_slab(v1*u.km/u.s, v2*u.km/u.s).moment0()

    plt.subplot(1,2,2)
    plt.imshow(m0.value, origin='lower')
    for name, src in sources.items():
        src['aperture'].plot(color='red', lw=1.5)
    plt.title(f'Scaled VLA truth M0 {tag}')
    plt.colorbar(label='mJy beam$^{-1}$ km s$^{-1}$')
    #plt.show()
    plt.savefig('fast_m0_vla_scaled_truth.png',dpi=150)

In [ ]:
outdir = './VLA/'
vla_data = vla_on_fast_cube.filled_data[:].value   # mJy/beam
vel = vla_on_fast_cube.spectral_axis.to(u.km/u.s).value

header_cube = vla_on_fast_cube.header.copy()
header_cube['BUNIT'] = 'mJy/beam'

nchan, ny, nx = vla_data.shape

truth_total_cube = np.zeros_like(vla_data, dtype=float)

for name, src in sources.items():

    spatial_mask = make_circular_mask_from_wcs(
        fast_wcs_2d,
        (ny, nx),
        src['ra'],
        src['dec'],
        src['radius_arcmin']
    )

    vel_mask = (vel >= src['vmin']) & (vel <= src['vmax'])

    comp_cube = np.zeros_like(vla_data, dtype=float)

    comp_cube[vel_mask, :, :] = (
        vla_data[vel_mask, :, :]
        * spatial_mask[None, :, :]
        * scale_factors[name]
    )

    truth_total_cube += comp_cube

    # 保存 source-specific scaled truth cube
    # cube_out = f'{outdir}/truth_{name}_scaled_VLA_cube.fits'
    # fits.writeto(
    #     cube_out,
    #     comp_cube.astype('float32'),
    #     header_cube,
    #     overwrite=True
    # )

    # 生成并保存 source-specific moment0 map
    # comp_cube unit = mJy/beam, dv ≈ 20 km/s
    dv = 20.0
    comp_m0 = np.nansum(comp_cube, axis=0) * dv

    header_m0 = vla_on_fast_cube[0].header.copy()
    header_m0['BUNIT'] = 'mJy/beam km/s'
    header_m0['COMMENT'] = f'Scaled VLA truth moment0 map for {name}.'
    header_m0['COMMENT'] = f'Velocity range: {src["vmin"]}-{src["vmax"]} km/s.'
    header_m0['COMMENT'] = f'Scale factor: {scale_factors[name]:.6f}.'

    m0_out = f'{outdir}/truth_{name}_scaled_VLA_M0.fits'
    fits.writeto(
        m0_out,
        comp_m0.astype('float32'),
        header_m0,
        overwrite=True
    )

    print(name)
    #print('  cube:', cube_out)
    print('  M0:  ', m0_out)
    print('  M0 peak:', np.nanmax(comp_m0))
    print('  M0 sum: ', np.nansum(comp_m0))

In [ ]:
# total moment0, 可先按 5200-5600 km/s 保存
vel_mask_total = (vel >= 5200) & (vel <= 5600)
truth_total_m0_5200_5600 = np.nansum(truth_total_cube[vel_mask_total, :, :], axis=0) * 20.0

header_m0_total = vla_on_fast_cube[0].header.copy()
header_m0_total['BUNIT'] = 'mJy/beam km/s'
header_m0_total['COMMENT'] = 'Total scaled VLA truth moment0 map.'
header_m0_total['COMMENT'] = 'Source-specific scale factors applied before summation.'
header_m0_total['COMMENT'] = 'Velocity range: 5200-5600 km/s.'

fits.writeto(
    f'{outdir}/truth_total_scaled_VLA_M0_5200_5600.fits',
    truth_total_m0_5200_5600.astype('float32'),
    header_m0_total,
    overwrite=True
)

## Test unconvolved VLA

In [ ]:
def make_circular_mask_from_wcs(wcs2d, shape, ra, dec, radius_arcmin):
    ny, nx = shape
    yy, xx = np.mgrid[:ny, :nx]

    coord = SkyCoord(ra*u.deg, dec*u.deg)
    x0, y0 = wcs2d.world_to_pixel(coord)

    pixscale = u.Quantity(wcs2d.proj_plane_pixel_scales(), u.deg)
    pixscale_x = np.abs(pixscale[0].to(u.arcmin).value)
    pixscale_y = np.abs(pixscale[1].to(u.arcmin).value)

    r_pix = radius_arcmin / np.sqrt(pixscale_x * pixscale_y)

    rr = np.sqrt((xx - x0)**2 + (yy - y0)**2)
    return rr <= r_pix

def beam_area_arcmin2(header):
    bmaj = abs(header['BMAJ']) * u.deg
    bmin = abs(header['BMIN']) * u.deg
    return (1.1331 * bmaj * bmin).to(u.arcmin**2).value

In [ ]:
for name, src in sources.items():

    v1 = src['vmin'] * u.km/u.s
    v2 = src['vmax'] * u.km/u.s

    cube_src = vla_masked.spectral_slab(v1, v2)

    # 原始 VLA moment0: Jy / VLA beam km/s
    m0 = cube_src.moment0()
    m0_data = m0.value
    m0_header = m0.header.copy()
    wcs2d = WCS(m0_header).celestial

    mask = make_circular_mask_from_wcs(
        wcs2d,
        m0_data.shape,
        src['ra'],
        src['dec'],
        src['radius_arcmin']
    )

    f = scale_factors[name]

    # 版本1：mJy / VLA beam km/s，只用于检查
    truth_mjy_per_vlabeam = m0_data * 1000.0 * f * mask

    h1 = m0_header.copy()
    h1['BUNIT'] = 'mJy/VLAbeam km/s'
    h1['COMMENT'] = f'Scaled original VLA truth map for {name}.'
    h1['COMMENT'] = 'Not convolved to FAST resolution.'
    h1['COMMENT'] = f'Scale factor = {f:.6f}.'

    fits.writeto(
        f'{outdir}/truth_{name}_scaled_VLA_M0_mJyPerVLAbeam.fits',
        truth_mjy_per_vlabeam.astype('float32'),
        h1,
        overwrite=True
    )

    # 版本2：mJy / arcmin^2 km/s，用于 beam convolution
    omega_vla = beam_area_arcmin2(m0_header)

    truth_mjy_per_arcmin2 = truth_mjy_per_vlabeam / omega_vla

    h2 = m0_header.copy()
    h2['BUNIT'] = 'mJy/arcmin2 km/s'
    h2['COMMENT'] = f'Scaled original VLA truth surface-brightness map for {name}.'
    h2['COMMENT'] = 'Use this map for convolution with FAST beam kernels.'
    h2['COMMENT'] = 'Not convolved to FAST resolution.'
    h2['COMMENT'] = f'VLA beam area = {omega_vla:.6e} arcmin2.'
    h2['COMMENT'] = f'Scale factor = {f:.6f}.'

    fits.writeto(
        f'{outdir}/truth_{name}_scaled_VLA_M0_mJyPerArcmin2.fits',
        truth_mjy_per_arcmin2.astype('float32'),
        h2,
        overwrite=True
    )

    print(name)
    print('  omega_vla [arcmin2] =', omega_vla)
    print('  peak mJy/VLAbeam =', np.nanmax(truth_mjy_per_vlabeam))
    print('  peak mJy/arcmin2 =', np.nanmax(truth_mjy_per_arcmin2))

In [ ]:
source_names = ['HCG100', 'NGC7810', 'MRK935', 'AGC105092']

# -----------------------------
# Sum mJy / VLA beam maps
# -----------------------------
total_vlabeam = None
header_vlabeam = None

for name in source_names:
    fname = f'{outdir}/truth_{name}_scaled_VLA_M0_mJyPerVLAbeam.fits'
    data = fits.getdata(fname)
    header = fits.getheader(fname)

    if total_vlabeam is None:
        total_vlabeam = np.zeros_like(data, dtype=float)
        header_vlabeam = header.copy()

    total_vlabeam += np.nan_to_num(data, nan=0.0)

header_vlabeam['BUNIT'] = 'mJy/VLAbeam km/s'
header_vlabeam['COMMENT'] = 'Total scaled original VLA truth map.'
header_vlabeam['COMMENT'] = 'Sum of source-specific truth maps.'

fits.writeto(
    f'{outdir}/truth_total_scaled_VLA_M0_mJyPerVLAbeam.fits',
    total_vlabeam.astype('float32'),
    header_vlabeam,
    overwrite=True
)

# -----------------------------
# Sum mJy / arcmin^2 maps
# -----------------------------
total_arcmin2 = None
header_arcmin2 = None

for name in source_names:
    fname = f'{outdir}/truth_{name}_scaled_VLA_M0_mJyPerArcmin2.fits'
    data = fits.getdata(fname)
    header = fits.getheader(fname)

    if total_arcmin2 is None:
        total_arcmin2 = np.zeros_like(data, dtype=float)
        header_arcmin2 = header.copy()

    total_arcmin2 += np.nan_to_num(data, nan=0.0)

header_arcmin2['BUNIT'] = 'mJy/arcmin2 km/s'
header_arcmin2['COMMENT'] = 'Total scaled original VLA truth surface-brightness map.'
header_arcmin2['COMMENT'] = 'Use this map for convolution with FAST beam kernels.'

fits.writeto(
    f'{outdir}/truth_total_scaled_VLA_M0_mJyPerArcmin2.fits',
    total_arcmin2.astype('float32'),
    header_arcmin2,
    overwrite=True
)

print('Saved total truth maps.')
print('Total mJy/VLAbeam peak:', np.nanmax(total_vlabeam))
print('Total mJy/arcmin2 peak:', np.nanmax(total_arcmin2))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(22, 4), constrained_layout=True)

vmax = np.nanpercentile(total_vlabeam[total_vlabeam > 0], 99.5)

for ax, name in zip(axs[:4], source_names):
    data = fits.getdata(f'{outdir}/truth_{name}_scaled_VLA_M0_mJyPerVLAbeam.fits')
    im = ax.imshow(data, origin='lower', vmin=0, vmax=vmax)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])

im = axs[4].imshow(total_vlabeam, origin='lower', vmin=0, vmax=vmax)
axs[4].set_title('Total')
axs[4].set_xticks([])
axs[4].set_yticks([])

cbar = fig.colorbar(im, ax=axs, shrink=0.85)
cbar.set_label('mJy / VLA beam km/s')

plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
vmax = np.nanpercentile(total_arcmin2[total_arcmin2 > 0], 99.5)

plt.imshow(total_arcmin2, origin='lower', vmin=0, vmax=vmax)
plt.title('Total scaled VLA truth map')
plt.colorbar(label='mJy arcmin$^{-2}$ km s$^{-1}$')
plt.xlabel('VLA pixel')
plt.ylabel('VLA pixel')
plt.show()

In [ ]:
truth_conv = fits.getdata('./VLA/truth_total_scaled_VLA_M0_5200_5600.fits')*(1.1331 * 2.9 * 2.9)
truth_unconv = fits.getdata('./VLA/truth_total_scaled_VLA_M0_mJyPerVLAbeam.fits')

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

im0 = axs[0].imshow(truth_conv, origin='lower')
axs[0].set_title('Convolved VLA truth')
plt.colorbar(im0, ax=axs[0])

im1 = axs[1].imshow(truth_unconv, origin='lower')
axs[1].set_title('Unconvolved VLA truth')
plt.colorbar(im1, ax=axs[1])


plt.tight_layout()
plt.show()

print('conv peak:', np.nanmax(truth_conv))
print('unconv peak:', np.nanmax(truth_unconv))
print('conv sum:', np.nansum(truth_conv))
print('unconv sum:', np.nansum(truth_unconv))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))



im0 = axs[0].imshow(fast_m0.value, origin='lower',vmin=0)
axs[0].set_title('FAST raw moment0')
plt.colorbar(im0, ax=axs[0])

#plt.subplot(1,2,2)

for tag, v1, v2 in [
    ('5200_5600', 5200, 5600),
]:
    m0 = truth_cube.spectral_slab(v1*u.km/u.s, v2*u.km/u.s).moment0()

    im1 = axs[1].imshow(m0.value, origin='lower')
    for name, src in sources.items():
        src['aperture'].plot(ax=axs[1],color='red', lw=1.5)
    axs[1].set_title(f'Scaled VLA M0 {tag}')
    plt.colorbar(im1, ax=axs[1],label='mJy beam$^{-1}$ km s$^{-1}$')
    #plt.show()
    
im2 = axs[2].imshow(truth_unconv, origin='lower')
axs[2].set_title('Unconvolved VLA truth')
plt.colorbar(im2, ax=axs[2],label='mJy beam$^{-1}$ km s$^{-1}$')
fig.savefig('fast_m0_vla_scaled_truth.pdf',dpi=150,bbox_inches='tight')

### Flux convolve vs. original

In [ ]:
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS

def beam_area_arcmin2(header):
    bmaj = abs(header['BMAJ']) * u.deg
    bmin = abs(header['BMIN']) * u.deg
    return (1.1331 * bmaj * bmin).to(u.arcmin**2).value

def pixel_area_arcmin2(header):
    w = WCS(header).celestial
    pixscale = u.Quantity(w.proj_plane_pixel_scales(), u.deg)
    pix_x = abs(pixscale[0].to(u.arcmin).value)
    pix_y = abs(pixscale[1].to(u.arcmin).value)
    return pix_x * pix_y

def flux_from_mjy_per_beam_map(data, header):
    """
    data: mJy/beam km/s
    output: Jy km/s
    """
    omega_pix = pixel_area_arcmin2(header)
    omega_beam = beam_area_arcmin2(header)
    flux_jykms = np.nansum(data) * omega_pix / omega_beam / 1000.0
    return flux_jykms

In [ ]:
def flux_from_surface_map(data, header):
    """
    data: mJy/arcmin^2 km/s
    output: Jy km/s
    """
    omega_pix = pixel_area_arcmin2(header)
    flux_jykms = np.nansum(data) * omega_pix / 1000.0
    return flux_jykms

In [ ]:
def flux_from_mjy_per_fastbeam_map(data, header, fast_fwhm_arcmin=4.0):
    """
    data: mJy/FAST beam km/s
    output: Jy km/s
    """
    omega_pix = pixel_area_arcmin2(header)
    omega_fast_beam = 1.1331 * fast_fwhm_arcmin * fast_fwhm_arcmin
    flux_jykms = np.nansum(data) * omega_pix / omega_fast_beam / 1000.0
    return flux_jykms

In [ ]:
# Unconvolved VLA-grid truth, mJy/arcmin^2 km/s
unconv_file = './VLA/truth_total_scaled_VLA_M0_mJyPerArcmin2.fits'
unconv_data = fits.getdata(unconv_file)
unconv_header = fits.getheader(unconv_file)

flux_unconv = flux_from_surface_map(unconv_data, unconv_header)

# Previous convolved truth on FAST grid, mJy/FAST beam km/s
conv_file = './VLA/truth_total_scaled_VLA_M0_5200_5600.fits'
conv_data = fits.getdata(conv_file)
conv_header = fits.getheader(conv_file)

flux_conv = flux_from_mjy_per_fastbeam_map(conv_data, conv_header, fast_fwhm_arcmin=2.9)

print('Unconvolved VLA-grid truth flux =', flux_unconv, 'Jy km/s')
print('Convolved FAST-grid truth flux  =', flux_conv, 'Jy km/s')
print('ratio unconv / conv =', flux_unconv / flux_conv)